# Cross-method leakage experiment (standalone)
Reuses the exact deterministic data-prep from the main pipeline, then trains **LitePhospho, DeepPhos, MusiteDeep** on a **random (leaky)** split and the **homology-controlled** split through the identical `train_eval_split`. Produces the real `fig_leakage_crossmethod.png` and the fidelity check. ~3-4 h; no ESM2, no full benchmark.


# LitePhospho v4 — Q1 evidence pipeline

**What changed vs v3 (and why it matters for a Q1 paper):**

1. **Homology-aware split (MMseqs2, 30% identity).** Proteins are clustered and whole clusters go to train/val/test. This is the leakage-free evaluation your proposal promised — and the central novelty of the paper.
2. **Baselines under the SAME split:** Logistic Regression, Linear SVM, Random Forest, XGBoost, plain CNN. A Q1 reviewer rejects without these.
3. **Compact 1D CNN (<1M params)** that preserves window position (flatten, not global pooling) — keeps it CPU-deployable while capturing position-specific motifs.
4. **AUPRC as the primary metric + ECE calibration** (your proposal, Limitation #6).
5. **Faithful explanations** via Integrated Gradients + a deletion test (not just pretty pictures).
6. **Green-computing metrics:** CPU latency, model size, energy + CO2 estimate.

Run top to bottom. Training the hero model is the only slow part (~1-2 h on T4).

In [ ]:
# LitePhospho v4 - LEAN + DETERMINISTIC Kaggle pipeline.
# No Colab, no checkpoints/resume, no memory cruft. Seeded for reproducible results.
import os
assert os.path.exists('/kaggle/input'), 'Kaggle-only notebook: attach the datasets first.'
print('Kaggle environment OK.')

Mounted at /content/drive


## Segment 1 — Paths

In [ ]:
from pathlib import Path
RAW_BASE  = Path('/kaggle/input/datasets/mdkarimulislam/bioinformatics-thesis-project/bio_project')
PTMD_PATH = Path('/kaggle/input/datasets/mdkarimulislam/total-ptm-disease/Total.txt')
WORK_BASE = Path('/kaggle/working')
EPSD_DIR = RAW_BASE/'epsd_human'; UNIPROT_DIR = RAW_BASE/'uniprot'
PROC_DIR = WORK_BASE/'data/processed'; SAVE_DIR = WORK_BASE
for d in [WORK_BASE, PROC_DIR]: d.mkdir(parents=True, exist_ok=True)
print('RAW_BASE :', RAW_BASE, '| exists:', RAW_BASE.exists())
print('SAVE_DIR :', SAVE_DIR, '(figures + FINAL files go here)')
print('PTMD     :', PTMD_PATH, '| exists:', PTMD_PATH.exists())

RAW_BASE    : /content/drive/MyDrive/LitePhospho/bio_project | exists: True
SAVE_DIR    : /content/drive/MyDrive/LitePhospho/Output
PTMD_PATH   : /content/drive/MyDrive/LitePhospho/TotalPTMDisease/Total.txt | exists: True


In [ ]:
# ===== DEFINITIONS — imports, constants, classes, helpers. NO heavy compute. =====
# Run this once near the top. On reconnect: [mount] -> [Paths] -> [this cell] -> [RESUME] -> continue.
import os, re, copy, time, gc, pickle
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             precision_score, recall_score, accuracy_score,
                             confusion_matrix, roc_curve, precision_recall_curve)

# ---- constants / encoders ----
WINDOW = 31; HALF = WINDOW // 2
AMINO_ACIDS = list('ACDEFGHIKLMNPQRSTVWYX')
vocab = {aa: i for i, aa in enumerate(AMINO_ACIDS)}
idx_to_aa = {i: aa for aa, i in vocab.items()}
PAD_IDX = vocab['X']; NAA = len(AMINO_ACIDS); BATCH = 512
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

def encode_window(w):
    return torch.tensor([vocab.get(a, PAD_IDX) for a in w.upper()], dtype=torch.long)

class PhosphoDataset(Dataset):
    def __init__(self, recs):
        self.X = torch.stack([encode_window(r[2]) for r in recs])
        self.y = torch.tensor([r[3] for r in recs], dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

# ---- model classes ----
class PlainCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.conv = nn.Conv1d(embed_dim, 128, 5, padding=2)
        self.fc1 = nn.Linear(128, 64); self.fc2 = nn.Linear(64, 1); self.drop = nn.Dropout(0.3)
    def forward(self, x, embedded=None):
        emb = embedded.transpose(1,2) if embedded is not None else self.embedding(x).transpose(1,2)
        h = F.relu(self.conv(emb)).max(dim=2).values
        h = self.drop(F.relu(self.fc1(h)))
        return self.fc2(h).squeeze(1)

class LitePhospho(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.c3 = nn.Conv1d(embed_dim, 64, 3, padding=1)
        self.c5 = nn.Conv1d(embed_dim, 64, 5, padding=2)
        self.c7 = nn.Conv1d(embed_dim, 64, 7, padding=3)
        self.bn1 = nn.BatchNorm1d(192)
        self.conv2 = nn.Conv1d(192, 64, 3, padding=1); self.bn2 = nn.BatchNorm1d(64)
        self.conv3 = nn.Conv1d(64, 32, 1); self.bn3 = nn.BatchNorm1d(32)
        self.fc1 = nn.Linear(32*WINDOW, 256); self.bn4 = nn.BatchNorm1d(256)
        self.fc2 = nn.Linear(256, 64); self.fc3 = nn.Linear(64, 1); self.drop = nn.Dropout(dropout)
    def _trunk(self, emb):
        h = torch.cat([F.relu(self.c3(emb)), F.relu(self.c5(emb)), F.relu(self.c7(emb))], dim=1)
        h = self.drop(self.bn1(h))
        h = self.drop(self.bn2(F.relu(self.conv2(h))))
        h = self.drop(self.bn3(F.relu(self.conv3(h))))
        h = h.flatten(1)
        x = self.drop(F.relu(self.bn4(self.fc1(h))))
        x = self.drop(F.relu(self.fc2(x)))
        return self.fc3(x).squeeze(1)
    def forward(self, x, embedded=None):
        emb = embedded.transpose(1,2) if embedded is not None else self.embedding(x).transpose(1,2)
        return self._trunk(emb)

class DeepPhosNet(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, growth=32, kernels=(1,3,5,7,9), dropout=0.4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.blocks = nn.ModuleList()
        for k in kernels:
            layers, in_c = nn.ModuleList(), embed_dim
            for _ in range(3):
                layers.append(nn.Conv1d(in_c, growth, k, padding=k//2)); in_c += growth
            self.blocks.append(layers)
        ch = len(kernels) * (embed_dim + 3*growth)
        self.reduce = nn.Conv1d(ch, 32, 1); self.bn = nn.BatchNorm1d(32)
        self.fc1 = nn.Linear(32*WINDOW, 128); self.fc2 = nn.Linear(128, 1); self.drop = nn.Dropout(dropout)
    def forward(self, x, embedded=None):
        emb = embedded.transpose(1,2) if embedded is not None else self.embedding(x).transpose(1,2)
        outs = []
        for layers in self.blocks:
            h = emb
            for conv in layers:
                h = torch.cat([h, F.relu(conv(h))], dim=1)
            outs.append(h)
        z = torch.cat(outs, dim=1)
        z = self.drop(F.relu(self.bn(self.reduce(z)))).flatten(1)
        z = self.drop(F.relu(self.fc1(z)))
        return self.fc2(z).squeeze(1)

class MusiteDeepNet(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, ch=128, heads=4, dropout=0.4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.conv1 = nn.Conv1d(embed_dim, ch, 5, padding=2)
        self.conv2 = nn.Conv1d(ch, ch, 3, padding=1); self.bn = nn.BatchNorm1d(ch)
        self.attn = nn.MultiheadAttention(ch, heads, dropout=dropout, batch_first=True)
        self.fc1 = nn.Linear(ch*WINDOW, 128); self.fc2 = nn.Linear(128, 1); self.drop = nn.Dropout(dropout)
    def forward(self, x, embedded=None):
        emb = embedded.transpose(1,2) if embedded is not None else self.embedding(x).transpose(1,2)
        h = self.bn(F.relu(self.conv2(F.relu(self.conv1(emb))))).transpose(1,2)
        a,_ = self.attn(h, h, h)
        z = self.drop(a).flatten(1)
        z = self.drop(F.relu(self.fc1(z)))
        return self.fc2(z).squeeze(1)

class ESMHead2(nn.Module):
    def __init__(self, d, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d,256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout),
                                 nn.Linear(256,64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(dropout),
                                 nn.Linear(64,1))
    def forward(self, x): return self.net(x).squeeze(1)

# ---- helper functions ----
results = {}
def record(name, yt, scores, preds):
    results[name] = {'ROC_AUC': roc_auc_score(yt, scores),
                     'PR_AUC': average_precision_score(yt, scores), 'F1': f1_score(yt, preds)}
    print(f"{name:22s} ROC-AUC={results[name]['ROC_AUC']:.4f}  PR-AUC={results[name]['PR_AUC']:.4f}  F1={results[name]['F1']:.4f}")

def deep_scores(model, loader):
    model.eval(); ys, ss = [], []
    with torch.no_grad():
        for bx, by in loader:
            ss.append(torch.sigmoid(model(bx.to(device))).cpu()); ys.append(by)
    return torch.cat(ys).numpy(), torch.cat(ss).numpy()

def onehot_matrix(recs):
    n = len(recs); X = np.zeros((n, WINDOW*NAA), np.float32); y = np.zeros(n, np.int64)
    for i, r in enumerate(recs):
        for j, aa in enumerate(r[2]): X[i, j*NAA + vocab.get(aa, PAD_IDX)] = 1.0
        y[i] = r[3]
    return X, y

def score_windows(win_list, bs=4096):
    hero.eval(); out = []
    with torch.no_grad():
        for k in range(0, len(win_list), bs):
            Xb = torch.stack([encode_window(w) for w in win_list[k:k+bs]]).to(device)
            out.append(torch.sigmoid(hero(Xb)).cpu().numpy())
    return np.concatenate(out) if out else np.array([])

def expected_calibration_error(y, p, bins=10):
    edges = np.linspace(0, 1, bins+1); ece = 0.0
    for i in range(bins):
        m = (p > edges[i]) & (p <= edges[i+1])
        if m.sum() > 0: ece += m.mean() * abs(y[m].mean() - p[m].mean())
    return ece

def train_model(model, epochs=60, lr=2e-4, wd=2e-3, patience=12, tag='model'):
    model = model.to(device); crit = nn.BCEWithLogitsLoss()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, 'min', patience=6, factor=0.5, min_lr=1e-6)
    best, wait, tr_hist, va_hist = float('inf'), 0, [], []; path = SAVE_DIR/f'best_{tag}.pt'
    for ep in range(1, epochs+1):
        model.train(); tot = 0.0
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device); opt.zero_grad()
            loss = crit(model(bx), by); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); tot += loss.item()*bx.size(0)
        tr = tot/len(train_ds); model.eval(); tot = 0.0
        with torch.no_grad():
            for bx, by in val_loader:
                bx, by = bx.to(device), by.to(device); tot += crit(model(bx), by).item()*bx.size(0)
        va = tot/len(val_ds); sched.step(va); tr_hist.append(tr); va_hist.append(va); flag = ''
        if va < best: best, wait = va, 0; torch.save(model.state_dict(), path); flag = '  <- best'
        else:
            wait += 1
            if wait >= patience: print(f'[{tag}] ep{ep:02d} tr={tr:.4f} va={va:.4f} early stop'); break
        print(f'[{tag}] ep{ep:02d} tr={tr:.4f} va={va:.4f} lr={opt.param_groups[0]["lr"]:.1e}{flag}')
    model.load_state_dict(torch.load(path, map_location=device)); return model, tr_hist, va_hist

def train_eval_split(ModelClass, tr, va, te, epochs=80, lr=3e-4, wd=1e-3, patience=12):
    trl = DataLoader(PhosphoDataset(tr), 512, shuffle=True, num_workers=2, pin_memory=True)
    vds = PhosphoDataset(va); vall = DataLoader(vds, 1024); tel = DataLoader(PhosphoDataset(te), 1024)
    m = ModelClass(NAA).to(device); crit = nn.BCEWithLogitsLoss()
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=wd); best, wait, sd = 1e9, 0, None
    for ep in range(1, epochs+1):
        m.train()
        for bx, by in trl:
            bx, by = bx.to(device), by.to(device); opt.zero_grad(); crit(m(bx), by).backward(); opt.step()
        m.eval(); tot = 0.0
        with torch.no_grad():
            for bx, by in vall: bx, by = bx.to(device), by.to(device); tot += crit(m(bx), by).item()*bx.size(0)
        vl = tot/len(vds)
        if vl < best: best, wait, sd = vl, 0, copy.deepcopy(m.state_dict())
        else:
            wait += 1
            if wait >= patience: break
    m.load_state_dict(sd); m.eval(); ys, ss = [], []
    with torch.no_grad():
        for bx, by in tel: ss.append(torch.sigmoid(m(bx.to(device))).cpu()); ys.append(by)
    y = torch.cat(ys).numpy(); s = torch.cat(ss).numpy()
    return roc_auc_score(y, s), average_precision_score(y, s), s, y

def train_esm_head(Xtr, ytr, Xva, yva, seed=42, epochs=100, patience=10):
    torch.manual_seed(seed)
    trl = DataLoader(TensorDataset(torch.from_numpy(Xtr.astype(np.float32)), torch.from_numpy(ytr)), 1024, shuffle=True)
    val = DataLoader(TensorDataset(torch.from_numpy(Xva.astype(np.float32)), torch.from_numpy(yva)), 2048)
    head = ESMHead2(Xtr.shape[1]).to(device); opt = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
    crit = nn.BCEWithLogitsLoss(); best, wait, sd = 1e9, 0, None
    for ep in range(1, epochs+1):
        head.train()
        for bx, by in trl:
            bx, by = bx.to(device), by.to(device); opt.zero_grad(); crit(head(bx), by).backward(); opt.step()
        head.eval(); tot = 0.0
        with torch.no_grad():
            for bx, by in val: bx, by = bx.to(device), by.to(device); tot += crit(head(bx), by).item()*bx.size(0)
        vl = tot/len(val.dataset)
        if vl < best: best, wait, sd = vl, 0, copy.deepcopy(head.state_dict())
        else:
            wait += 1
            if wait >= patience: break
    head.load_state_dict(sd); return head

print('definitions loaded — classes and helpers ready.')

# ===== Reproducibility: deterministic seeding (fixes run-to-run variation) =====
import os as _os, random as _random
def seed_all(seed=42):
    _os.environ['PYTHONHASHSEED'] = str(seed)
    _random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
seed_all(42)
print('deterministic seeding ON (cudnn.deterministic=True, benchmark=False, seed=42)')


device: cuda
definitions loaded — classes and helpers ready.


## Segment 2 — Parse EPSD + UniProt FASTA

In [ ]:
import pandas as pd, re, numpy as np

df = pd.read_csv(EPSD_DIR / 'Homo sapiens.txt', sep=None, engine='python')
print('EPSD shape:', df.shape, '| columns:', list(df.columns))

acc_regex = re.compile(r'^[OPQ][0-9][A-Z0-9]{3}[0-9](?:-\d+)?$|^[A-NR-Z][0-9]{5}(?:-\d+)?$', re.I)
def pick_acc_column(df):
    best_col, best_hits = None, -1
    for c in df.columns:
        hits = df[c].astype(str).str.strip().str.match(acc_regex).sum()
        if hits > best_hits:
            best_hits, best_col = hits, c
    return best_col
acc_col = pick_acc_column(df)
pos_col = next(c for c in df.columns if 'position' in c.lower())
res_col = next(c for c in df.columns if c.lower() in {'residue','aa','modified residue'})

epsd = df[[acc_col, res_col, pos_col]].copy()
epsd.columns = ['acc_raw', 'residue', 'pos']
epsd['acc']     = epsd['acc_raw'].astype(str).str.split('-').str[0]
epsd['pos']     = epsd['pos'].astype(int)
epsd['residue'] = epsd['residue'].astype(str).str.upper()
epsd = epsd[epsd['residue'].isin(list('STY'))].drop_duplicates(['acc','pos']).copy()
print('EPSD S/T/Y rows after de-dup:', len(epsd))

EPSD shape: (1071725, 6) | columns: ['EPSD ID', 'UniProt ID', 'AA', 'Position', 'Source', 'Reference']
EPSD S/T/Y rows after de-dup: 871982


In [ ]:
seq_dict = {}
with open(UNIPROT_DIR / 'uniprotkb.fasta') as f:
    acc, buf = None, []
    for line in f:
        line = line.strip()
        if line.startswith('>'):
            if acc:
                seq_dict[acc] = ''.join(buf)
            parts = line[1:].split('|')
            acc = parts[1] if len(parts) >= 2 else parts[0].split()[0]
            buf = []
        else:
            buf.append(line)
    if acc:
        seq_dict[acc] = ''.join(buf)
print(f'Loaded {len(seq_dict):,} UniProt sequences')

Loaded 20,405 UniProt sequences


## Segment 3 — Build windows (positives + 1:1 negatives)

In [ ]:
WINDOW = 31
HALF   = WINDOW // 2

epsd_known = epsd[epsd['acc'].isin(seq_dict)].copy()
epsd_known['pos0'] = epsd_known['pos'] - 1

pos_records = []
for acc, grp in epsd_known.groupby('acc'):
    seq    = seq_dict[acc]
    padded = 'X'*HALF + seq + 'X'*HALF
    for p, r in zip(grp['pos0'].astype(int).values, grp['residue'].values):
        if 0 <= p < len(seq) and seq[p] == r:   # FIX: residue must match EPSD annotation
            w = padded[p:p+WINDOW]
            if len(w) == WINDOW:
                pos_records.append((acc, p, w, 1))
print(f'Positive windows: {len(pos_records):,}')

pos_set      = {(a, p) for a, p, _, _ in pos_records}
pos_proteins = {a for a, _, _, _ in pos_records}

neg_records = []
for acc in pos_proteins:
    seq    = seq_dict[acc]
    padded = 'X'*HALF + seq + 'X'*HALF
    for i, aa in enumerate(seq):
        if aa in 'STY' and (acc, i) not in pos_set:
            w = padded[i:i+WINDOW]
            if len(w) == WINDOW:
                neg_records.append((acc, i, w, 0))
print(f'Negative windows (raw): {len(neg_records):,}')

rng = np.random.default_rng(42)
neg_idx = rng.choice(len(neg_records), size=min(len(pos_records), len(neg_records)), replace=False)
neg_bal = [neg_records[i] for i in neg_idx]
all_records = pos_records + neg_bal
print(f'Balanced total: {len(all_records):,}  (pos {len(pos_records):,} / neg {len(neg_bal):,})')

Positive windows: 664,562
Negative windows (raw): 1,270,424
Balanced total: 1,329,124  (pos 664,562 / neg 664,562)


## Segment 4 — Homology-aware split (MMseqs2, 30% identity)

**This is the core of the paper.** We cluster every protein at 30% sequence identity and
assign whole clusters to train/val/test, so no test protein is >30% similar to any training
protein. This is what 'leakage-free' actually means — a plain protein-level split does NOT
guarantee it (paralogs/homologs leak).

In [ ]:
import os, subprocess

# One-time download of the MMseqs2 static binary
if not os.path.exists('mmseqs/bin/mmseqs'):
    os.system('wget -q https://mmseqs.com/latest/mmseqs-linux-avx2.tar.gz && tar xzf mmseqs-linux-avx2.tar.gz')
MMSEQS = os.path.abspath('mmseqs/bin/mmseqs')
print('MMseqs2 binary:', MMSEQS, '| exists:', os.path.exists(MMSEQS))

# Write FASTA of proteins that have at least one positive site
clust_fasta = str(PROC_DIR / 'pos_proteins.fasta')
with open(clust_fasta, 'w') as f:
    for acc in sorted(pos_proteins):
        f.write(f'>{acc}\n{seq_dict[acc]}\n')

out_prefix = str(PROC_DIR / 'clust')
tmp_dir    = '/tmp/mmseqs_tmp' # Changed to a local temporary directory
os.makedirs(tmp_dir, exist_ok=True) # Ensure the temporary directory exists
cmd = f'{MMSEQS} easy-cluster {clust_fasta} {out_prefix} {tmp_dir} --min-seq-id 0.3 -c 0.5 --cov-mode 1 -v 1'
print('Running:', cmd)

# Use subprocess.run to capture output and errors for better debugging
try:
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, check=True)
    print("MMseqs2 stdout:", result.stdout)
    if result.stderr:
        print("MMseqs2 stderr:", result.stderr)
except subprocess.CalledProcessError as e:
    print(f"MMseqs2 command failed with exit code {e.returncode}")
    print(f"stdout: {e.stdout}")
    print(f"stderr: {e.stderr}")
    # Re-raise or handle the error as appropriate for your workflow
    raise

prot2cluster = {}
with open(out_prefix + '_cluster.tsv') as f:
    for line in f:
        rep, mem = line.strip().split('\t')
        prot2cluster[mem] = rep
clusters = sorted(set(prot2cluster.values()))
print(f'{len(pos_proteins):,} proteins -> {len(clusters):,} clusters at 30% identity')


MMseqs2 binary: /content/mmseqs/bin/mmseqs | exists: True
Running: /content/mmseqs/bin/mmseqs easy-cluster /content/drive/MyDrive/LitePhospho/Output/data/processed/pos_proteins.fasta /content/drive/MyDrive/LitePhospho/Output/data/processed/clust /tmp/mmseqs_tmp --min-seq-id 0.3 -c 0.5 --cov-mode 1 -v 1
MMseqs2 stdout: 
19,671 proteins -> 9,894 clusters at 30% identity


In [ ]:
from sklearn.model_selection import train_test_split

train_cl, temp_cl = train_test_split(clusters, test_size=0.30, random_state=42)
val_cl,   test_cl = train_test_split(temp_cl,   test_size=0.50, random_state=42)
train_cl, val_cl, test_cl = set(train_cl), set(val_cl), set(test_cl)

def split_of(acc):
    c = prot2cluster.get(acc)
    if c in train_cl: return 'train'
    if c in val_cl:   return 'val'
    if c in test_cl:  return 'test'
    return None

buckets = {'train': [], 'val': [], 'test': []}
for r in all_records:
    s = split_of(r[0])
    if s:
        buckets[s].append(r)
train_recs, val_recs, test_recs = buckets['train'], buckets['val'], buckets['test']

def pos_frac(recs):
    return np.mean([r[3] for r in recs]) if recs else 0
print(f'Train: {len(train_recs):,} (pos {pos_frac(train_recs):.2%})')
print(f'Val  : {len(val_recs):,} (pos {pos_frac(val_recs):.2%})')
print(f'Test : {len(test_recs):,} (pos {pos_frac(test_recs):.2%})')
print('Cluster overlap check (must all be 0):',
      len(train_cl & val_cl), len(train_cl & test_cl), len(val_cl & test_cl))

Train: 906,460 (pos 50.21%)
Val  : 214,758 (pos 50.46%)
Test : 207,906 (pos 48.60%)
Cluster overlap check (must all be 0): 0 0 0


## The experiment: random vs homology, all three models


In [ ]:
# Capacity-dependent leakage: same clean windows, random (leaky) vs homology split, identical trainer.
# The leaky ROC also serves as the FIDELITY check (should approach each method's published ~0.84).
import numpy as np, torch, json, matplotlib.pyplot as plt

# matched RANDOM (leaky) split of the same balanced windows
all_recs = train_recs + val_recs + test_recs
rng = np.random.default_rng(42)
perm = rng.permutation(len(all_recs)); n_tr, n_va = len(train_recs), len(val_recs)
rnd_tr = [all_recs[i] for i in perm[:n_tr]]
rnd_va = [all_recs[i] for i in perm[n_tr:n_tr+n_va]]
rnd_te = [all_recs[i] for i in perm[n_tr+n_va:]]
print(f'Random split : {len(rnd_tr):,}/{len(rnd_va):,}/{len(rnd_te):,}')
print(f'Homology split: {len(train_recs):,}/{len(val_recs):,}/{len(test_recs):,}')

MODELS = [('LitePhospho', LitePhospho), ('DeepPhos', DeepPhosNet), ('MusiteDeep', MusiteDeepNet)]
res = {}
print(f'\n{"model":12s} {"params":>9s} {"leaky ROC":>10s} {"honest ROC":>11s} {"drop":>8s}')
for name, Cls in MODELS:
    params = sum(p.numel() for p in Cls(NAA).parameters())
    seed_all(42); roc_r, pr_r, _, _ = train_eval_split(Cls, rnd_tr, rnd_va, rnd_te)
    seed_all(42); roc_h, pr_h, _, _ = train_eval_split(Cls, train_recs, val_recs, test_recs)
    res[name] = dict(params=int(params), random=float(roc_r), homology=float(roc_h),
                     drop=float(roc_r-roc_h), pr_random=float(pr_r), pr_homology=float(pr_h))
    print(f'{name:12s} {params:>9,} {roc_r:>10.3f} {roc_h:>11.3f} {roc_r-roc_h:>+8.3f}')

print('\n--- Fidelity check (leaky ROC should approach published ~0.84 for DeepPhos/MusiteDeep) ---')
for n in res:
    print(f'  {n:12s} leaky ROC {res[n]["random"]:.3f}')
print('--- Leakage drop vs params (does drop scale with capacity?) ---')
for n in sorted(res, key=lambda k: res[k]['params']):
    print(f'  {n:12s} params {res[n]["params"]:>8,}  drop {res[n]["drop"]:+.3f}')

# figure (neutral title; interpret capacity framing after seeing the numbers)
nm = list(res); rnd_v = [res[n]['random'] for n in nm]; hom_v = [res[n]['homology'] for n in nm]
x = np.arange(len(nm)); w = 0.35; plt.figure(figsize=(7,5))
plt.bar(x-w/2, rnd_v, w, label='Random split (leaky)', color='#95a5a6')
plt.bar(x+w/2, hom_v, w, label='Homology 30% (honest)', color='#e74c3c')
for i in range(len(nm)):
    plt.text(i, hom_v[i]-0.03, f'-{rnd_v[i]-hom_v[i]:.3f}', ha='center', fontweight='bold', color='white')
plt.xticks(x, nm); plt.ylim(0.6, 0.9); plt.ylabel('ROC-AUC')
plt.title('Leakage inflation: random vs homology-controlled split')
plt.legend(); plt.tight_layout()
plt.savefig(SAVE_DIR/'fig_leakage_crossmethod.png', dpi=200, bbox_inches='tight'); plt.close()
json.dump(res, open(SAVE_DIR/'leakage_crossmethod.json', 'w'), indent=2)
print('\nSaved fig_leakage_crossmethod.png + leakage_crossmethod.json (REAL clean values) ->', SAVE_DIR)
